# ModernBERT smell-token classifier inference (batched, routed heads)

Produces JSONL outputs keyed by `id` so you can join sidecar later.

In [ ]:
from pathlib import Path
import os
import json
import torch

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / 'training_inference').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing training_inference/.')

REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
print('REPO_ROOT =', REPO_ROOT)

from training_inference.data_utils import (
    create_dataset_build, create_tokenizer, PretokenizedSmellDataset, TokenClassificationCollator
)
from training_inference.train_loop import make_dataloader
from training_inference.checkpoint_utils import load_training_checkpoint, build_model_from_checkpoint_metadata, validate_checkpoint_smell_schema
from training_inference.inference_utils import run_batched_inference, save_inference_outputs_jsonl, preview_inference_records


In [ ]:
# ==== User-editable settings ====
SEED = 42
BACKBONE_NAME_OVERRIDE = None  # keep None to use checkpoint metadata
MAX_LENGTH = 8192
BATCH_SIZE = 1
NUM_WORKERS = 0
USE_AMP = True

DATASET_CONFIG_PATH = REPO_ROOT / 'training_inference' / 'dataset_paths.config46AndComplexity.json'
RUN_DIR = REPO_ROOT / 'training_inference' / 'runs' / 'modernbert_routed_dynamic_heads'
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / 'best.pt'
SPLIT_TO_RUN = 'test'  # 'train' | 'val' | 'test'
OUTPUT_JSONL = RUN_DIR / f'inference_{SPLIT_TO_RUN}.jsonl'
RETURN_PROBABILITIES = False
PROBABILITY_TOPK = 3  # only used when RETURN_PROBABILITIES=True


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
ckpt = load_training_checkpoint(CHECKPOINT_PATH, map_location='cpu')
print('ckpt keys:', ckpt.keys())


In [ ]:
# Rebuild dataset mappings/splits deterministically with the same seed.
# Note: if you changed split settings in training notebook, match them here.
dataset_build = create_dataset_build(
    repo_root=REPO_ROOT,
    dataset_config_path=DATASET_CONFIG_PATH,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=SEED,
    train_neg_pos_ratio=None,  # no rebalancing for inference splits
    balance_per_smell=False,
    split_manifest_dir=None,
)

# Use checkpoint label/smell mappings if available (safer for consistency).
label_to_id = ckpt['label_to_id']
id_to_label = {int(v): k for k, v in label_to_id.items()}
smell_to_head = ckpt['smell_to_head']

# Validate exact smell routing schema against current dataset config
validate_checkpoint_smell_schema(ckpt, dataset_build.smell_maps.smell_to_head)


In [ ]:
tokenizer_name = BACKBONE_NAME_OVERRIDE or ckpt.get('tokenizer_name_or_path') or 'answerdotai/ModernBERT-base'
tokenizer = create_tokenizer(tokenizer_name, use_fast=True, truncation_side='right')

# Build dataset object for the chosen split using checkpoint mappings (for exact routing ids)
from training_inference.data_utils import LabelMaps, SmellMaps
label_maps = LabelMaps(label_to_id=label_to_id, id_to_label=id_to_label, ignore_index=-100)
smell_maps = SmellMaps(smell_to_head=smell_to_head, head_to_smell={int(v): k for k, v in smell_to_head.items()})

if SPLIT_TO_RUN == 'train':
    split_examples = dataset_build.train_examples
elif SPLIT_TO_RUN == 'val':
    split_examples = dataset_build.val_examples
elif SPLIT_TO_RUN == 'test':
    split_examples = dataset_build.test_examples
else:
    raise ValueError(SPLIT_TO_RUN)

run_ds = PretokenizedSmellDataset(split_examples, tokenizer, label_maps, smell_maps, max_length=MAX_LENGTH)
collator = TokenClassificationCollator(tokenizer=tokenizer, label_pad_id=-100)
run_loader = make_dataloader(run_ds, collator, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [ ]:
model = build_model_from_checkpoint_metadata(ckpt, backbone_name=BACKBONE_NAME_OVERRIDE)
model.to(device)
print('Loaded model with', len(smell_to_head), 'heads and', len(label_to_id), 'labels')


In [ ]:
records = run_batched_inference(
    model=model,
    dataloader=run_loader,
    device=device,
    id_to_label=id_to_label,
    routed_only=True,
    return_probabilities=RETURN_PROBABILITIES,
    probability_topk=PROBABILITY_TOPK,
    amp_enabled=USE_AMP,
)

save_inference_outputs_jsonl(records, OUTPUT_JSONL)
print('saved:', OUTPUT_JSONL)
print('num records:', len(records))
print('truncated count:', sum(int(r['was_truncated']) for r in records))
preview_inference_records(records, n=2)


**Sidecar placeholder**

This notebook intentionally does **not** read `role_section_excerpts.sidecar.jsonl` yet.
Join later by `id` when you add source trace-back / pointer reconstruction.